# RankSEG × MMSegmentation (argmax vs RankSEG on test pipeline)

This Colab demonstrates how to use `rankseg.mmseg` compatibility helpers to compare
`argmax` and `RankSEG` predictions on MMSegmentation outputs.

In [ ]:
!pip -q install uv
!uv pip install --system openmim mmengine
!uv pip install --system 'mmcv>=2.0.0'
!uv pip install --system git+https://github.com/open-mmlab/mmsegmentation.git
!uv pip install --system git+https://github.com/Leev1s/rankseg.git@codex/explore-integration-options-with-mmsegmentation

In [ ]:
import torch
from mmseg.apis import init_model, inference_model
from rankseg.mmseg import restore_semantic_probs_from_mmseg, postprocess_mmseg

## 1) Run MMSeg inference and keep `SegDataSample`

In [ ]:
config_path = 'configs/fcn/fcn_r50-d8_4xb4-80k_ade20k-512x512.py'
checkpoint_path = 'https://download.openmmlab.com/mmsegmentation/v0.5/fcn/fcn_r50-d8_512x512_80k_ade20k/fcn_r50-d8_512x512_80k_ade20k_20200614_144016-f8ac5082.pth'
img_path = 'demo/demo.png'

model = init_model(config_path, checkpoint_path, device='cuda:0')
data_sample = inference_model(model, img_path)
print(type(data_sample))

## 2) Convert to probabilities + get argmax baseline and RankSEG output

In [ ]:
probs = restore_semantic_probs_from_mmseg(data_sample)  # (1, C, H, W)
argmax_pred = probs.argmax(dim=1)                      # (1, H, W)
rankseg_pred = postprocess_mmseg(
    data_sample,
    rankseg_kwargs=dict(metric='dice', solver='RMA', output_mode='multiclass')
)
print('argmax:', argmax_pred.shape, 'rankseg:', rankseg_pred.shape)

## 3) Compare metrics (example utilities)

In [ ]:
def mean_iou(pred, target, num_classes, ignore_index=255):
    pred = pred.view(-1)
    target = target.view(-1)
    valid = target != ignore_index
    pred = pred[valid]
    target = target[valid]
    ious = []
    for c in range(num_classes):
        p = pred == c
        t = target == c
        inter = (p & t).sum().item()
        union = (p | t).sum().item()
        if union > 0:
            ious.append(inter / union)
    return sum(ious) / max(len(ious), 1)

# Suppose gt is available from your test loader / data_sample.gt_sem_seg
# gt = data_sample.gt_sem_seg.data.squeeze(0)
# mIoU_argmax = mean_iou(argmax_pred.squeeze(0), gt, num_classes=probs.shape[1])
# mIoU_rankseg = mean_iou(rankseg_pred.squeeze(0), gt, num_classes=probs.shape[1])
# print({'argmax': mIoU_argmax, 'rankseg': mIoU_rankseg, 'delta': mIoU_rankseg - mIoU_argmax})

## 4) Hook into `tools/test.py` workflow

In real experiments, integrate comparison inside evaluator / test loop:
1. keep MMSeg `SegDataSample` outputs in test loop
2. compute baseline (`argmax`) from `seg_logits`
3. compute `RankSEG` prediction via `postprocess_mmseg`
4. log both metric sets and deltas in the same run